In [1]:
import sys, os; sys.path.insert(0, os.path.join("..", "src")) if os.path.basename(os.getcwd())=="notebooks" else sys.path.insert(0, "src")
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import config, data_layer, econometrics as ek, outputs, ministry_spec as msp
pd.set_option("display.width", 220)
FR = "FR13"

# FR13 — «Nazirlik spesifikasiyası» işlək ssenari kimi

Bu dəftər Nazirliyin öz makroekonometrik nümunə-modelinin **92 sətirlik tənlik kataloqunu**
paketin daxilində **ikinci, işlək spesifikasiya** kimi həll edir: onun öz əmsalları, öz dəyişən
adları, birləşdirilmiş məlumat qatı üzərində. Bu, modelin əvəzlənməsi deyil — **ayrıca
ssenaridir**; nəticələri eyni «Geriyə Sınaq» cədvəlində, eyni təsadüfi gəzişmə (RW) etalonuna
qarşı və eyni pəncərədə qiymətləndirilir.

Yazdığı sətirlər üç əhatə ilə məhdudlaşır: `forecast_long.csv`-də `fr=FR13`, `source=ministry_spec`;
`validation_backtest.csv`-də `model=NAZIRLIK_SPES`; `equations_catalog.csv`-də `fr=FR13`.

## 1. Tələb və status

Kataloqun 92 sətri sətir-sətir təsnifata əsasən dörd səbətə bölünür və bu bölgü burada koddan
YENİDƏN hesablanır (sayımlar sənəddəki ilə eyni çıxmalıdır):

| Səbət | İzah |
|---|---|
| **A — 67 sətir** | birləşdirilmiş məlumat qatında təqdim olunduğu kimi işləyir |
| **B — 14 sətir** | sənədləşdirilmiş əvəzedici və ya qısa/birləşdirilmiş nümunə ilə işləyir (bayraqlı) |
| **C — 8 sətir** | kataloqda tərifi verilməyən xəta-korreksiyası səviyyəsini gözləyir |
| **D — 3 sətir** | dörd tədiyə balansı dəyişəninin (`RACBAR`, `MSC`, `IFOC`, `OI_L_OS`) tərifini gözləyir |

İki fərdi hal ayrıca göstərilir. **`ECM_MS_TP1`** kataloqda təyin olunmayıb; qərar (D3/R17) onu
`ECM_MS_TP` ilə eyni qəbul etməyə və nəticəni **bayraqlamağa** icazə verir — qalan yeddi səviyyə
üçün sətir «tərif gözlənilir» statusunda qalır. **49-cu sətir** EViews sintaksisi deyil, mətn
kimi çıxarılıb («before 2005 * 0.0419645 …») — statusu «mətn dəqiqləşdirilməlidir».

In [2]:
rows = msp.load_catalog()
print("kataloq sətirləri:", len(rows), "| sintaksis təhlili keçən:", sum(r.parses for r in rows))
bsum = msp.bucket_summary(rows)
display(bsum)
assert list(bsum["sətir"]) == [67, 14, 8, 3], "səbət sayımı təsnifatla uyğun gəlmir"
assert list(bsum["davranış tənliyi"]) == [55, 11, 8, 3], "davranış tənliyi sayımı uyğun gəlmir"
print("\nstatus bölgüsü:")
display(msp.catalog_frame(rows)["status"].value_counts().rename_axis("status").to_frame("sətir"))

kataloq sətirləri: 92 | sintaksis təhlili keçən: 91


,səbət,izah,sətir,davranış tənliyi,işə salınır
0,A,birləşdirilmiş məlumat qatında təqdim olunduğu...,67,55,66
1,B,sənədləşdirilmiş əvəzedici və ya qısa/birləşdi...,14,11,14
2,C,kataloqda tərifi verilməyən xəta-korreksiyası ...,8,8,1
3,D,tədiyə balansı dəyişənlərinin tərifini gözləyir,3,3,0



status bölgüsü:


,sətir
status,
işə salınır,66
işə salınır — əvəzedici/qısa nümunə (bayraqlı),14
tərif gözlənilir,7
bloklanıb — dəyişən tərifi yoxdur,3
işə salınır — ECM ekvivalentliyi FƏRZ EDİLİB (bayraqlı),1
mətn dəqiqləşdirilməlidir,1


In [3]:
# 49-cu sətir, bloklanmış sətirlər və bayraqlı ECM — açıq şəkildə göstərilir
tab = msp.catalog_frame(rows)
display(tab[tab["№"].isin([3, 8, 15, 16, 49])][["№", "hədəf", "status", "qeyd"]])
print("\ntərif gözləyən 7 səviyyə:")
display(tab[tab["status"] == msp.STATUS_C][["№", "hədəf", "qeyd"]])

,№,hədəf,status,qeyd
2,3,MS_TP,işə salınır — ECM ekvivalentliyi FƏRZ EDİLİB (...,`ECM_MS_TP1` kataloqda təyin olunmayıb; D3/R17...
7,8,OI_L_OS,bloklanıb — dəyişən tərifi yoxdur,tərifi verilməyən dəyişən(lər): OI_L_OS — dörd...
14,15,MSC,bloklanıb — dəyişən tərifi yoxdur,"tərifi verilməyən dəyişən(lər): IFOC, MSC — dö..."
15,16,RACBAR,bloklanıb — dəyişən tərifi yoxdur,tərifi verilməyən dəyişən(lər): RACBAR — dördü...
48,49,RVA_CONST,mətn dəqiqləşdirilməlidir,"kataloq sətri EViews sintaksisi deyil, MƏTN ki..."



tərif gözləyən 7 səviyyə:


,№,hədəf,qeyd
8,9,MGO,gözlənilən səviyyə tərifi: ECM_MGO12
12,13,MGNO,gözlənilən səviyyə tərifi: ECM_MGNO_NEW
22,23,RIEA,gözlənilən səviyyə tərifi: ECM_RIEANEW2020
29,30,RIDNS,gözlənilən səviyyə tərifi: ECM_RIDNS2020
34,35,RVA_ACCOM,gözlənilən səviyyə tərifi: ECM_RVA_ACCOM2020
74,75,W_STMANAG,gözlənilən səviyyə tərifi: ECM_W_STMANAG2020
80,81,RIEA,gözlənilən səviyyə tərifi: ECM_RIEA_NEW_2020


### 1.1 Rejim dummilərinin aktivliyi

Sabit əmsallı ssenaridə hər rejim dummisinin mövcud nümunə dövründə **həqiqətən dəyişib-dəyişmədiyi**
yoxlanılmalıdır: heç vaxt 1 olmayan dummi tənliyə heç nə vermir, həmişə 1 olan dummi isə faktiki
olaraq sabitə çevrilir. Yoxlama fərziyyə ilə deyil, panelin faktı ilə aparılır — hər tənlik üçün
nümunənin başlanğıcı onun öz dəyişənlərinin ən gec başlayan sırasından götürülür.

**Nəticə mühümdür və ilkin təsnifatın gözləntisini dəqiqləşdirir.** İş kitabları qoşulduqdan sonra
milli hesablar, tədiyə balansı və monetar sıralar 1995/1996-cı illərə qədər uzanır, ona görə
1996–1998-ci illərə aid dummilərin böyük hissəsi **aktiv qalır**; yalnız 25-ci sətirdəki
`(T>1998)` termi bütün nümunə boyu 1 olur, yəni sabitə çevrilir. Fəaliyyət növləri üzrə blokda
(61–90-cı sətirlər) isə sıralar 1999/2001-dən başlayır və ən erkən rejim həddi `@BEFORE("2004")`-dür:
o, nümunənin yalnız ilk beş ilini (1999–2003) əhatə edir.

In [4]:
dum = msp.inactive_pre_sample_dummies(rows)
print("dummisi butun numune boyu sabit qalan setir sayi:", len(dum))
display(dum)

bdum = msp.branch_dummy_table(rows)
print("fealiyyet novleri bloku - numune baslangici ve en erken rejim heddi:")
display(bdum)
print("blokda en erken rejim heddi:", int(bdum["ən erkən rejim həddi"].min()),
      "| numunenin en erken baslangici:", int(bdum["nümunənin başlanğıcı"].min()))

dummisi butun numune boyu sabit qalan setir sayi: 1


,№,hədəf,nümunənin başlanğıcı,həmişə 0 (qeyri-aktiv),həmişə 1 (sabitə çevrilir)
0,25,RHC,1999,—,(T>1998)


fealiyyet novleri bloku - numune baslangici ve en erken rejim heddi:


,№,hədəf,nümunənin başlanğıcı,ən erkən rejim həddi,həmin rejimin əhatə etdiyi il
0,61,W_AGR,2001,NaN,NaN
1,62,W_MINE,1999,NaN,NaN
2,63,W_MANU,1999,NaN,NaN
3,64,W_ELEC,1999,2004.0,5.0
4,65,W_WATE,1999,NaN,NaN
5,66,W_CONST,1999,NaN,NaN
6,67,W_TRADE,1999,NaN,NaN
7,68,W_ACCOM,1999,2006.0,7.0
8,69,W_TRANS,2001,NaN,NaN
9,70,W_INFORM,2001,NaN,NaN


blokda en erken rejim heddi: 2004 | numunenin en erken baslangici: 1997


## 2. Məlumat qatı — Nazirliyin öz iş kitabları, yalnız oxunan rejimdə

Ssenarinin tarixi paneli Nazirliyin nümunə-model iş kitablarından (`MOE SOCIAL`, `MOE SNA`,
`MOE INF`, `MOE BOP`, `INDUSTRY`, `MOE T&C`, `MOE FISCAL`) çıxarılıb və paketin giriş qovluğunda
donmuş şəkildə saxlanılır (`data/moe_spec_panel.csv`, provenans `data/moe_spec_provenance.csv`).
Hər sətir üçün iş kitabı, vərəq, sətir nömrəsi və **gözlənilən etiket** yazılıb; `data_layer`-in
`verify_moe_spec_panel()` funksiyası fayllar əlçatan olduqda paneli mənbədən yenidən qurub
tutuşdurur.

Üç mühafizə qaydası koda daxildir:

1. **yalnız faktiki sütunlar (≤2024)** — illik başlığın üstündəki annotasiya sətri 2025-i
   «gözlənilən», 2026–2030-u «proqnoz» kimi işarələyir;
2. **`model` və `add factor` sətirləri oxunmur** — onlar nümunə modelin öz proqnozu və mülahizə
   düzəlişidir, müşahidə deyil;
3. **`PBCRHCF` 2020-dən 2030-a qədər −0,0047-də dondurulub** — həmin illər tarix olmadığı üçün
   sıra yalnız ≤2019 oxunur.

Panelin müstəqil çarpaz yoxlaması: onların əvvəlki çıxarılışındakı 74 sıra (`moe_extract/*.csv`)
ilə maksimal fərq 1e-6-dan kiçikdir; `DEF_REST` və `DEF_FINANCE` törəmələri 2015-ci ildə dəqiq
1,0000 verir (2024: 1,9913 və 1,4293).

In [5]:
prov = data_layer.moe_spec_provenance()
panel_df = data_layer.moe_spec_panel()
print("panel:", len(panel_df), "sətir |", panel_df["var"].nunique(), "sıra |",
      f"{panel_df.year.min()}–{panel_df.year.max()}")
display(prov.groupby("workbook").agg(sıra=("var", "size"), ilk=("year_first", "min"),
                                     son=("year_last", "max")))
print("\nqısa nümunəli sıralar (n < 20) — bunlar B səbətinin bayraqlarını izah edir:")
display(prov[prov["n"] < 20][["var", "workbook", "sheet", "row", "year_first", "year_last", "n", "note_az"]])

panel: 3910 sətir | 150 sıra | 1995–2024


,sıra,ilk,son
workbook,,,
INDUSTRY.xlsx,2,2009,2024
MOE BOP.xlsx,6,1996,2024
MOE FISCAL.xlsx,1,2005,2015
MOE INF.xlsx,23,1995,2024
MOE SNA.xlsx,49,1995,2024
MOE SOCIAL.xlsx,67,1995,2024
MOE T&C.xlsx,2,2004,2024



qısa nümunəli sıralar (n < 20) — bunlar B səbətinin bayraqlarını izah edir:


,var,workbook,sheet,row,year_first,year_last,n,note_az
35,LP_PROF,MOE SOCIAL.xlsx,add8,19,2010,2024,15,fəaliyyət növü üzrə əmək məhsuldarlığının real...
38,LP_ADMN,MOE SOCIAL.xlsx,add8,20,2010,2024,15,fəaliyyət növü üzrə əmək məhsuldarlığının real...
41,LP_REST,MOE SOCIAL.xlsx,add8,21,2010,2024,15,fəaliyyət növü üzrə əmək məhsuldarlığının real...
50,LP_HEALTH,MOE SOCIAL.xlsx,add8,24,2010,2024,15,fəaliyyət növü üzrə əmək məhsuldarlığının real...
53,LP_ART,MOE SOCIAL.xlsx,add8,25,2010,2024,15,fəaliyyət növü üzrə əmək məhsuldarlığının real...
56,LP_OTHER,MOE SOCIAL.xlsx,add8,26,2010,2024,15,fəaliyyət növü üzrə əmək məhsuldarlığının real...
96,VA_ADMN,MOE SNA.xlsx,sosial və digər xidmətlər,35,2009,2024,16,"inzibati xidmətlər, əlavə dəyər (2009-dan — QI..."
115,FOREIGNTURIST,MOE SNA.xlsx,IN,26,2006,2024,19,xaricdən gələn turistlərin sayı (2006-dan — QI...
145,PI_IND3_1,INDUSTRY.xlsx,3.1,4,2009,2024,16,"qida məhsulları istehsalı, müqayisəli qiymətlə..."
146,PI_IND3_2,INDUSTRY.xlsx,3.2. VAR,4,2009,2024,16,"içki istehsalı, müqayisəli qiymətlərlə (2009-d..."


In [6]:
# eynilik yoxlamaları: donmuş panel ↔ mənbə (fayl varsa) və törəmələrin baza ili
diff = data_layer.verify_moe_spec_panel()
print("mənbədən təkrar oxu:", "fayl yoxdur (təhvil mühiti)" if diff is None else f"maks. fərq = {diff:.3e}")

hist = msp.history_panel()
for nm, num, den in (("DEF_REST", "VA_REST", "RVA_REST"), ("DEF_FINANCE", "VA_FINANCE", "RVA_FINANCE")):
    v = hist[num][2015] / hist[den][2015]
    assert abs(v - 1.0) < 1e-9, f"{nm}: 2015 bazası 1.0 deyil ({v})"
    print(f"{nm}: 2015 = {v:.6f}  ·  2024 = {hist[num][2024]/hist[den][2024]:.4f}")
assert max(hist["PBCRHC"]) == 2019, "PBCRHC mühafizəsi pozulub (2020+ oxunmamalıdır)"
print("PBCRHC dövrü:", min(hist["PBCRHC"]), "–", max(hist["PBCRHC"]), "(2020+ dondurulmuş, oxunmur)")

mənbədən təkrar oxu: fayl yoxdur (təhvil mühiti)
DEF_REST: 2015 = 1.000000  ·  2024 = 1.9913
DEF_FINANCE: 2015 = 1.000000  ·  2024 = 1.4293
PBCRHC dövrü: 1995 – 2019 (2020+ dondurulmuş, oxunmur)


### 2.1 Dəyişən xəritəsi: alias və törəmələr

Kataloqun bəzi adları iş kitabında ayrıca sıra deyil. İki hal var və hər ikisi bayraqlanır:
**alias** (eyni sıranın ikinci adı) və **törəmə** (bir sətirlik açıq hesablama). `CPICMTP` və
`TIKINTI` adları iş kitablarında yoxdur — birincisi `cpimtp` ilə, ikincisi tikintinin real əlavə
dəyəri ilə eyniləşdirilir; hər ikisi görüş sualı kimi qeyd olunur.

In [7]:
al = pd.DataFrame([{"ad": k, "mənbə": v[0], "qeyd": v[1]} for k, v in data_layer.MOE_SPEC_ALIAS.items()])
de = pd.DataFrame([{"ad": k, "düstur": v[0], "qeyd": v[1]} for k, v in data_layer.MOE_SPEC_DERIVED.items()])
display(al); display(de)
print(msp.EQUATION_UNIT_FIX_NOTE)

,ad,mənbə,qeyd
0,WAGE,W,kataloq eyni sıranı iki adla çağırır
1,MINWAGE,MW,kataloq eyni sıranı iki adla çağırır
2,CPICMTP,CPIMTP,kataloqda yalnız bir sətirdə (17) rast gəlinir...
3,TIKINTI,RVA_CONST,kataloqun 51-ci sətri tikintinin real əlavə də...


,ad,düstur,qeyd
0,RW,W / CPI,real əmək haqqı — iş kitabında ayrıca sıra yoxdur
1,DEF_FINANCE,VA_FINANCE / RVA_FINANCE,maliyyə sektorunun deflyatoru — nominal ÷ sabi...
2,DEF_REST,VA_REST / RVA_REST,daşınmaz əmlak deflyatoru — nominal ÷ sabit (2...
3,RIO,IO / DEFID,"neft-qaz investisiyası, sabit qiymətlərlə — da..."
4,RINO,INO / DEFID,"qeyri-neft investisiyası, sabit qiymətlərlə — ..."
5,REALFINALC,FINALC_NOM / (HC / RHC),son istehlakın real həcmi — ev təsərrüfatların...


43-cü sətir: `NIRD` onluq kəsrə çevrilir (÷100) — nümunədaxili uyğunluq bunu birmənalı göstərir (OMFX 202.1 % → 3.1 %); əmsallara toxunulmur


## 3. Kod portu — nə götürülüb, nə dəqiqləşdirilib

Sintaksis analizatoru və blok-rekursiv
Gauss–Seidel həlledicisi EViews-tipli tənlik oxuyucusu metodikası əsasında burada yazılıb. Kataloqun faktiki sətirləri üzərində **dörd dəqiqləşdirmə**
aparılıb və hamısı koda şərh kimi yazılıb:

| № | Dəqiqləşdirmə | Nəyə görə |
|---|---|---|
| 1 | mötərizəsiz `T=1998` kimi dummilər mötərizəyə alınır | Python-da müqayisə toplamadan zəif bağlandığı üçün 40, 42 və 17-ci sətirlərdə **bütün sağ tərəf** tək müqayisəyə çevrilirdi və tənlik səssizcə sıfır artım verirdi |
| 2 | dəyişən adları böyük-kiçik hərfdən asılı olmadan həll olunur | kataloq `ecm_rxgno` təyin edir, `ECM_RXGNO` istinad edir (11, 44, 85-ci sətirlər) |
| 3 | `@SUM(ifadə)` dövr verilmədikdə bütün nümunə üzrə cəmlənir | 34-cü sətirdə `@SUM(OPWTI*(T=2015))` — əks halda sıfıra bölmə |
| 4 | sətir sabitləri dəyişən axtarışından çıxarılır | `@DURING("2004_2008")` içindəki `_2008` dəyişən adı sayılırdı |

Bundan əlavə, sol tərəfi **nisbət** olan tənliklər (17, 18, 22-ci sətirlər: `DLOG(MGNO/CPICMTP)`,
`M1/MB`) həll edilir: əvvəlcə nisbətin səviyyəsi tapılır, sonra hədəf dəyişən nisbətdən geri
çıxarılır (ifadə hədəf üzrə xəttidir).

**Ekzogen yolların mənbəyi dəyişdirilib.** Əvvəlki yanaşma sürücüləri təsadüfi sabitlərlə
uzadırdı (m2 6,644; brent ~69; tərəfdaş ~2,5). Bu modul onların yerinə **bu paketin nəşr olunmuş
proqnoz yollarını** qoyur — beləliklə iki spesifikasiya **eyni ekzogen fərziyyələr** üzərində
işləyir və fərq yalnız tənliklərdən gəlir.

## 4. Geriyə doğru sınaq — «Geriyə Sınaq» cədvəlinin ikinci sütunu

Sınaq bizim öz harness-imizin konvensiyası ilə aparılır: bir addımlıq (h = 1), genişlənən
pəncərə, sürücülər həmin ildə faktiki (şərti proqnoz), etalon təsadüfi gəzişmədir.

**Metodoloji güzəşt açıq elan olunur.** Nazirliyin əmsalları tam nümunə üzərində
qiymətləndirilib, ona görə onların proqnozu **nümunədaxili əmsallarla** hesablanır; bizim
tənliklərimiz isə hər vintajda yenidən qiymətləndirilir. Bu, müqayisəni **onların xeyrinə**
meyilləndirir. Üstəlik, hər sıra üçün bizim **ən yaxşı** formamız götürülür. Onların
spesifikasiyası bu iki güzəştə baxmayaraq bir sıra göstəricidə üstün çıxırsa, nəticə daha
möhkəmdir.

**Tərif yoxlaması.** Hər cüt üçün onların faktiki sırası bizim sınaq faylındakı `actual` sütunu
ilə tutuşdurulur (fərq/standart kənarlaşma). Uyğunsuzluq 1,0-dan böyükdürsə cüt müqayisədən
çıxarılır — belə hallar tərif fərqidir, bacarıq fərqi deyil.

In [8]:
hist = msp.history_panel()
for y in range(1996, config.LAST_ACTUAL):
    msp.refresh_ecm(hist, rows, y)
skill, bt_rows = msp.side_by_side(hist, rows)
display(skill)

,sıra kodu,Nazirlik №,hədəf,pəncərə,n,RMSE Nazirlik,RMSE bizim,RMSE RW,bizim model,bacarıq Nazirlik %,bacarıq bizim %,qeyd
0,cpi_infl,21,CPI,2012–2024,13,5.252414,4.676395,5.377369,AR1,2.3,13.0,tərif uyğunluğu: fərq/std = 0.00
1,d_agri,53,FPI_AZ,2017–2024,8,3.527169,1.654391,7.035006,YENI,49.9,76.5,tərif uyğunluğu: fərq/std = 0.04
2,d_constr,54,DEF_CONSTR,2009–2024,16,3.968045,5.233667,10.618947,YENI,62.6,50.7,tərif uyğunluğu: fərq/std = 0.04
3,d_tourism,55,DEF_ACCOM,2009–2024,16,16.377858,8.527270,17.842266,KOHNE,8.2,52.2,tərif uyğunluğu: fərq/std = 0.00
4,d_tourism,60,DEF_ACCOM,2009–2024,16,8.717614,8.527270,17.842266,KOHNE,51.1,52.2,tərif uyğunluğu: fərq/std = 0.00
5,d_transport,56,DEF_TRANSP,2022–2024,3,NaN,NaN,NaN,NaN,NaN,NaN,ortaq pəncərə 4 ildən qısadır
6,exp_goods_nonoil,11,RXGNO,2017–2024,8,615.193003,272.532315,361.327777,TƏRƏFDAŞ+BRENT,-70.3,24.6,tərif uyğunluğu: fərq/std = 0.00
7,exp_goods_nonoil,44,RXGNO,2017–2024,8,615.193003,272.532315,361.327777,TƏRƏFDAŞ+BRENT,-70.3,24.6,tərif uyğunluğu: fərq/std = 0.00
8,exp_goods_nonoil,46,RXGNO,2017–2024,8,255.474006,272.532315,361.327777,TƏRƏFDAŞ+BRENT,29.3,24.6,tərif uyğunluğu: fərq/std = 0.00
9,exp_goods_nonoil,52,RXGNO,2017–2024,8,243.694897,272.532315,361.327777,TƏRƏFDAŞ+BRENT,32.6,24.6,tərif uyğunluğu: fərq/std = 0.00


In [9]:
scored = skill.dropna(subset=["RMSE Nazirlik"]).copy()
win_rw = int((scored["bacarıq Nazirlik %"] > 0).sum())
win_us = int((scored["bacarıq Nazirlik %"] > scored["bacarıq bizim %"]).sum())
print(f"qiymətləndirilən tənlik sətri : {len(scored)}  ({scored['sıra kodu'].nunique()} göstərici)")
print(f"RW etalonunu üstələyən        : {win_rw}")
print(f"bizim ən yaxşı formamızı üstələyən: {win_us}")
print(f"müqayisə edilə bilməyən sətir : {len(skill) - len(scored)}")

qiymətləndirilən tənlik sətri : 21  (14 göstərici)
RW etalonunu üstələyən        : 15
bizim ən yaxşı formamızı üstələyən: 6
müqayisə edilə bilməyən sətir : 2


In [10]:
# `validation_backtest.csv`-yə YALNIZ `model=NAZIRLIK_SPES` sətirləri yazılır
assert set(bt_rows["model"]) == {"NAZIRLIK_SPES"}, "yad model adı ilə sətir yazıla bilməz"
written = outputs.write_backtest(bt_rows)
print("yazılan sətir:", len(bt_rows), "| fayldakı cəmi:", len(written))
print("NAZIRLIK_SPES sıraları:", sorted(bt_rows["series_code"].unique()))

yazılan sətir: 164 | fayldakı cəmi: 6747
NAZIRLIK_SPES sıraları: ['cpi_infl', 'd_agri', 'd_constr', 'd_tourism', 'exp_goods_nonoil', 'exp_serv_telecom', 'final_consumption', 'g_constr', 'g_tourism', 'g_trade', 'imp_goods_nonoil', 'imp_serv_transport', 'imp_serv_travel', 'inv_defl_g']


## 5. Ssenarinin həlli, 2025–2030

Tarix paneli 2024-də bitir. Ekzogen sıralar 2025-dən etibarən **bu paketin öz yolları** ilə
uzadılır; xəritədə qarşılığı olmayan sıralar üçün susma qaydası tətbiq olunur və hər sətir üçün
hansı qaydanın işlədiyi aşağıdakı cədvəldə göstərilir. Sonra tənliklər sistemi hər il üçün
blok-rekursiv Gauss–Seidel üsulu ilə həll olunur; partlayan yol mühafizəsi |Δln| > 0,30 olan
həlli qəbul etmir və həmin sətri jurnala yazır.

In [11]:
res = msp.run_scenario()
print("ekzogen sıra:", len(res.exog), "| endogen hədəf:", len(res.endo), "| həll olunan:", len(res.solved))
display(res.exog_log[res.exog_log["qayda"] != "susma"].reset_index(drop=True))
print("\nsusma qaydası ilə uzadılan sıra sayı:", int((res.exog_log["qayda"] == "susma").sum()))

ekzogen sıra: 106 | endogen hədəf: 55 | həll olunan: 51


,dəyişən,qayda,mənbə,qeyd
0,AGRI,gd:d_agri,proqnoz:g_agri,kənd təsərrüfatı əlavə dəyəri (real × deflyator)
1,CPIMTP,g,fərziyyə:import_price_infl,tərəfdaş İQİ idxal qiymət yolu ilə
2,ER,direct,fərziyyə:usd_azn,AZN/USD məzənnəsi FR10 rejim qaydasından
3,FOREIGNTURIST,g,proqnoz:g_tourism,turist sayı turizm sahəsinin real artımı ilə
4,HI,gd:cpi_infl,proqnoz:income_realg,əhalinin gəlirləri (real × inflyasiya)
5,IEA,gd:cpi_infl,proqnoz:income_realg,"sahibkarlıq gəlirləri, nominal"
6,INFLATION,direct,fərziyyə:cpi_infl,inflyasiya faizi birbaşa FR9 yoludur
7,L,g,proqnoz:employment_g,məşğulluq FR8 artım tempindən
8,LCPI_HP,loq-artım,fərziyyə:cpi_infl,loq-İQİ trendi inflyasiya yolu ilə uzadılır
9,MW,g,fərziyyə:cpi_infl,minimum əmək haqqı inflyasiya templə (fərziyyə)



susma qaydası ilə uzadılan sıra sayı: 73


In [12]:
if len(res.solve_log):
    lg = res.solve_log.groupby(["№", "hədəf"]).size().rename("rədd edilmiş il-keçid").reset_index()
    print("partlayan yol mühafizəsinin işə düşdüyü tənliklər:")
    display(lg)
else:
    print("partlayan yol qeydə alınmayıb")

partlayan yol mühafizəsinin işə düşdüyü tənliklər:


,№,hədəf,rədd edilmiş il-keçid
0,11,RXGNO,6
1,28,RVA_CONST,9
2,39,CI,3
3,44,RXGNO,6
4,57,DEF_NTP,2
5,65,W_WATE,2
6,72,W_PROF,2
7,85,L_MINE,18
8,91,TRANS_P,3


## 6. Proqnoz yolları və müqayisə

Ssenarinin yolları `forecast_long.csv`-yə `source=ministry_spec` etiketi ilə yazılır — bu, baza
proqnoz deyil, müqayisə sütunudur. Aşağıdakı qrafik hər göstərici üçün iki yolu yan-yana verir:
bu paketin mərkəzi yolu (`ours`) və Nazirlik spesifikasiyası ssenarisi.

In [13]:
fc = msp.forecast_rows(res, fr=FR)
print("proqnoz sətirləri:", len(fc), "|", fc["series_code"].nunique(), "göstərici")
written_fc = outputs.write_forecast_long(fc)
print("forecast_long.csv-də cəmi sətir:", len(written_fc))
print("mənbə bölgüsü:")
display(written_fc.groupby("source").size().rename("sətir").to_frame())

proqnoz sətirləri: 80 | 16 göstərici


forecast_long.csv-də cəmi sətir: 14603
mənbə bölgüsü:


,sətir
source,
imf_reference,15
ministry_spec,80
ours,13312
template_sample,1196


In [14]:
fl = pd.read_csv(os.path.join(config.OUT, "forecast_long.csv"), float_precision="round_trip")
ours_fc = fl[(fl["source"] == "ours") & (fl["kind"] == "forecast")]
codes = sorted(fc["series_code"].unique())
ncol, nrow = 4, int(np.ceil(len(codes) / 4))
fig, axes = plt.subplots(nrow, ncol, figsize=(15, 2.6 * nrow))
axes = np.atleast_1d(axes).ravel()
for ax, c in zip(axes, codes):
    a = ours_fc[ours_fc["series_code"] == c].sort_values("year")
    b = fc[fc["series_code"] == c].sort_values("year")
    if len(a):
        ax.plot(a["year"], a["value"], marker="o", lw=1.6, label="bu paketin modeli")
    ax.plot(b["year"], b["value"], marker="s", lw=1.6, ls="--", label="Nazirlik spesifikasiyası")
    ax.set_title(c, fontsize=9); ax.grid(alpha=.3); ax.tick_params(labelsize=8)
for ax in axes[len(codes):]:
    ax.axis("off")
axes[0].legend(fontsize=8)
fig.suptitle("2026–2030: bu paketin mərkəzi yolu ilə Nazirlik spesifikasiyası ssenarisinin müqayisəsi", fontsize=11)
fig.tight_layout()
os.makedirs(config.FIG, exist_ok=True)
fig.savefig(os.path.join(config.FIG, "fr13_ssenari_muqayise.png"), dpi=130, bbox_inches="tight")
plt.close(fig)
print("qrafik yazıldı: fr13_ssenari_muqayise.png")

qrafik yazıldı: fr13_ssenari_muqayise.png


In [15]:
# rəqəmlə müqayisə cədvəli (2026 və 2030)
cmp = []
for c in codes:
    a = ours_fc[ours_fc["series_code"] == c].set_index("year")["value"]
    b = fc[fc["series_code"] == c].set_index("year")["value"]
    cmp.append({"sıra kodu": c,
                "bizim 2026": (round(float(a.get(2026)), 2) if 2026 in a.index else None),
                "Nazirlik 2026": (round(float(b.get(2026)), 2) if 2026 in b.index else None),
                "bizim 2030": (round(float(a.get(2030)), 2) if 2030 in a.index else None),
                "Nazirlik 2030": (round(float(b.get(2030)), 2) if 2030 in b.index else None)})
display(pd.DataFrame(cmp))

,sıra kodu,bizim 2026,Nazirlik 2026,bizim 2030,Nazirlik 2030
0,cpi_infl,5.30,3.01,5.93,4.18
1,d_agri,5.79,9.71,6.11,10.25
2,d_constr,2.74,6.88,3.00,6.76
3,d_tourism,6.83,0.30,7.48,1.79
4,d_transport,2.63,7.16,2.54,8.08
5,exp_goods_nonoil,3679.85,3638.04,4672.99,6606.87
6,exp_serv_telecom,127.98,134.39,116.58,194.61
7,final_consumption,85099.08,7.05,135572.44,7.28
8,g_constr,3.28,1.25,4.18,2.01
9,g_tourism,15.57,10.07,14.57,9.55


## 7. Tənliklər kataloqu

Kataloqun bütün 92 sətri `equations_catalog.csv`-yə `fr=FR13` əhatəsi ilə əlavə olunur: tənliyin
adı `NAZIRLIK №nn` prefiksi daşıyır, `description` sütununda isə sətrin statusu və bayraq qeydi
saxlanılır. Beləliklə ekranlarda hər iki spesifikasiya eyni cədvəldən oxunur, lakin mənbələri
qarışmır.

In [16]:
cat = []
for r in rows:
    cat.append({"workbook": r.workbook, "sheet": r.sheet,
                "eq_name": f"NAZIRLIK №{r.no:02d} — {r.eq_name.split(',')[0] or r.target}",
                "description": f"[{r.bucket}] {r.status} — {r.description or r.target}. {r.note_az}",
                "lhs": r.lhs, "rhs": r.rhs, "series_code": r.target, "fr": FR,
                "sample_start": None, "sample_end": None, "adj_r2": None, "se_regression": None})
cat = pd.DataFrame(cat)
cat["sample_start"] = 1995; cat["sample_end"] = data_layer.MOE_LAST_ACTUAL
w = outputs.write_equations_catalog(cat)
print("FR13 sətirləri:", int((w["fr"] == FR).sum()), "| kataloqda cəmi:", len(w))

FR13 sətirləri: 92 | kataloqda cəmi: 486


## 8. Yekun yoxlamalar

Dəftər öz sətirlərindən kənara çıxmır: `forecast_long.csv`-də yalnız `fr=FR13` və
`source=ministry_spec`, `validation_backtest.csv`-də yalnız `model=NAZIRLIK_SPES`,
`equations_catalog.csv`-də yalnız `fr=FR13`. Aşağıdakı yoxlamalar bunu təsdiqləyir və
hər hansı pozuntuda dəftər dayanır.

In [17]:
fl = pd.read_csv(os.path.join(config.OUT, "forecast_long.csv"), float_precision="round_trip")
bt = pd.read_csv(os.path.join(config.OUT, "validation_backtest.csv"), float_precision="round_trip")
eq = pd.read_csv(os.path.join(config.OUT, "equations_catalog.csv"))

sub = fl[fl["fr"] == FR]
assert set(sub["source"]) == {"ministry_spec"}, "FR13 yalnız `ministry_spec` mənbəsi ilə yazmalıdır"
assert set(sub["kind"]) == {"forecast"}, "FR13 faktiki sətir yazmır"
assert sub["year"].min() >= config.FIRST_FORECAST and sub["year"].max() <= config.LAST_FORECAST
assert not fl[(fl["source"] == "ministry_spec") & (fl["fr"] != FR)].shape[0], "başqa FR-də `ministry_spec` sətri var"
assert set(bt[bt["model"] == "NAZIRLIK_SPES"]["series_code"]) <= set(bt["series_code"]), "yad sıra kodu"
assert int((eq["fr"] == FR).sum()) == 92, "kataloqda 92 sətir olmalıdır"
assert "ministry_spec" in outputs.SOURCE_LABELS_AZ, "mənbə lüğəti genişləndirilməyib"

print("FR13 proqnoz sətirləri      :", len(sub))
print("NAZIRLIK_SPES sınaq sətirləri:", int((bt['model'] == 'NAZIRLIK_SPES').sum()))
print("FR13 kataloq sətirləri      :", int((eq['fr'] == FR).sum()))
print("bütün yoxlamalar keçdi")

FR13 proqnoz sətirləri      : 80
NAZIRLIK_SPES sınaq sətirləri: 164
FR13 kataloq sətirləri      : 92
bütün yoxlamalar keçdi


## 9. Nəticə və görüş sualları

Ssenari **işləkdir**: 92 sətirdən 81-i həll oluna bilən statusdadır (67 A + 14 B), bir sətir
bayraqlı ECM ekvivalentliyi ilə əlavə olunur, yeddi sətir tərif gözləyir, üç sətir dörd tədiyə
balansı dəyişəninin tərifini gözləyir, bir sətir isə mətn kimi dəqiqləşdirilməlidir.

Görüşə çıxarılan altı dəqiqləşdirmə sualı:

1. **Yeddi xəta-korreksiyası səviyyəsinin tərifi** — `ECM_MGNO_NEW`, `ECM_MGO12`,
   `ECM_RIDNS2020`, `ECM_RIEANEW2020`, `ECM_RIEA_NEW_2020`, `ECM_RVA_ACCOM2020`,
   `ECM_W_STMANAG2020`; `ECM_MS_TP1` hazırda `ECM_MS_TP` ilə eyni qəbul edilib (bayraqlı).
2. **Dörd tədiyə balansı dəyişəni** — `RACBAR`, `MSC`, `IFOC`, `OI_L_OS`: tənlikləri `eq14`
   vərəqindədir, məlumat sətri yoxdur.
3. **49-cu sətrin mətni** — sintaksis şəklində təkrar təqdim olunması.
4. **İki ad** — `CPICMTP` (`cpimtp` ilə eyniləşdirilib) və `TIKINTI` (tikintinin real əlavə
   dəyəri ilə eyniləşdirilib) üçün təsdiq.
5. **43-cü sətrin `NIRD` vahidi** — tənlik faiz dərəcəsini onluq kəsr kimi işlədir, iş kitabının
   `Mon` vərəqi isə faiz vahidindədir; uyğunlaşdırma nümunədaxili uyğunluqla aparılıb.
6. **`RGDPO` və `VA_DIGER` təriflərinin təsdiqi** — hər ikisi hazırda sənədləşdirilmiş
   əvəzedici ilə işləyir.